# Session 3 Demo: From Pseudocode to Code — General Search, BFS & DFS

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualization library  ─  run this cell first
# ─────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
import fai_viz
print("✅ fai_viz visualization library loaded")

## Section 1: The General-Search Pseudocode (from slides)

```
Function General-Search(p, QUEUING-FN) returns a solution or failure
    frontier = Make-Queue(Make-Node(Initial-State[p]))
    Loop do
        If frontier is empty then return failure
        node = Remove-Front(frontier)
        If Goal-Test[p] on State(node) succeeds then return node
        frontier = QUEUING-FN(frontier, Expand(node, Actions[p]))
    End
```

**Key insight:** `QUEUING-FN` determines the search strategy:
- **BFS** → add to END (FIFO) → `ENQUEUE-AT-END`
- **DFS** → add to FRONT (LIFO) → `ENQUEUE-AT-FRONT`
- **UCS** → add sorted by cost → `ENQUEUE-BY-COST`

### 📌 Reference: BFS vs DFS — Search Tree Expansion

The diagram below shows how BFS and DFS differ on a **binary tree**. Numbers indicate the **order of node expansion**. Run this cell to see the contrast clearly.

- **BFS**: expands all nodes at depth 1 before going to depth 2 (→ shortest path first)
- **DFS**: dives as deep as possible before backtracking (→ less memory, not shortest)

In [ ]:
# Show the BFS vs DFS search tree reference figure
fai_viz.show_bfs_dfs_tree()

## Section 2: Problem Formulation & Representation

In [ ]:
from collections import deque

# ── Grid problem ─────────────────────────────────────────────────────────
def make_grid_problem(initial, goal, grid_size=5, walls=None):
    return {'initial': initial, 'goal': goal,
            'grid_size': grid_size, 'walls': set(walls or [])}

def is_goal(problem, state):
    return state == problem['goal']

def get_actions(problem, state):
    row, col = state
    size, walls = problem['grid_size'], problem.get('walls', set())
    return [(n, s) for n, s in
            [('up',(row-1,col)),('down',(row+1,col)),
             ('left',(row,col-1)),('right',(row,col+1))]
            if 0<=s[0]<size and 0<=s[1]<size and s not in walls]

def action_cost(problem, state, action):
    return 1   # uniform grid

# ── Node helpers ──────────────────────────────────────────────────────────
def make_node(state, parent=None, action=None, cost=0, depth=0):
    return {'state':state,'parent':parent,'action':action,'cost':cost,'depth':depth}

def reconstruct_path(node):
    path = []
    while node: path.append(node['state']); node = node['parent']
    return list(reversed(path))

def expand(problem, node, get_actions_fn, action_cost_fn):
    """Generate child nodes using provided get_actions and action_cost functions."""
    children = []
    for action_name, next_state in get_actions_fn(problem, node['state']):
        cost = action_cost_fn(problem, node['state'], action_name)
        children.append(make_node(next_state, node, action_name,
                                  node['cost'] + cost, node['depth'] + 1))
    return children

def is_cycle(node):
    """Return True if node's state already appears in its ancestor chain.
    Prevents DFS from looping along a single path (e.g. A→B→A→B→...)
    without needing a global visited set or an arbitrary depth limit."""
    ancestor = node['parent']
    while ancestor is not None:
        if ancestor['state'] == node['state']:
            return True
        ancestor = ancestor['parent']
    return False

# Demo
p = make_grid_problem((0,0), (4,4))
root = make_node((0,0))
print("Actions from (0,0):", get_actions(p, (0,0)))
print("Expand (0,0):", [(c['action'], c['state']) for c in expand(p, root, get_actions, action_cost)])

## Section 3: Implementing General-Search in Python

**Key insight:** The ONLY difference between BFS and DFS is HOW we add nodes to the frontier!

In [ ]:
def general_search(problem, queuing_fn, get_actions_fn, action_cost_fn,
                   is_goal_fn=None):
    """
    General Search algorithm with customizable queuing function, action generator, and cost function.
    """
    if is_goal_fn is None: is_goal_fn = is_goal
    frontier = deque([make_node(problem['initial'])])        # Make-Queue
    while True:                                              # Loop do
        if not frontier: return None                         #   If frontier is empty → failure
        node = frontier.popleft()                            #   node = Remove-Front
        if is_goal_fn(problem, node['state']): return node  #   If Goal-Test succeeds → return
        queuing_fn(frontier, expand(problem, node, get_actions_fn, action_cost_fn))  # QUEUING-FN

## Section 4: BFS — ENQUEUE-AT-END

BFS adds new nodes to the END of the queue → explores level by level (shallowest first)

In [ ]:
def enqueue_at_end(frontier, children):
    """BFS: ENQUEUE-AT-END — new nodes go to the BACK of the queue."""
    for child in children:
        frontier.append(child)

# Run BFS on 5x5 grid
p = make_grid_problem((0,0), (4,4))
print('=== BFS (ENQUEUE-AT-END) ===')
bfs_result = general_search(p, enqueue_at_end, get_actions, action_cost)
bfs_path = reconstruct_path(bfs_result)
print('Path:', bfs_path)


## Section 5: DFS — ENQUEUE-AT-FRONT

DFS adds new nodes to the FRONT of the queue → explores deepest branch first

**Cycle prevention belongs in the QUEUING-FN, not in `general_search`.**

`general_search` is a faithful translation of the course pseudocode — it stays generic. The DFS queuing function takes responsibility for filtering out children that would create a cycle on the current path, using `is_cycle(child)` before inserting each child.

This way BFS (`enqueue_at_end`) stays completely unchanged, and the cycle-check logic is co-located with the strategy that actually needs it.

In [ ]:
def enqueue_at_front(frontier, children):
    """DFS: ENQUEUE-AT-FRONT — new nodes go to the FRONT (LIFO).
    is_cycle filters out any child whose state already appears on the current path."""
    for child in reversed(children):
        if not is_cycle(child):          # skip if state already on this path
            frontier.appendleft(child)

# Run DFS
print('=== DFS (ENQUEUE-AT-FRONT) ===')
dfs_result = general_search(p, enqueue_at_front, get_actions, action_cost)
dfs_path = reconstruct_path(dfs_result)
print('Path:', dfs_path)

## Section 6: BFS vs DFS — Comparison & Path Visualisation

In [ ]:
# ── Compare BFS vs DFS ────────────────────────────────────────────────────
def search_with_count(problem, queuing_fn, get_actions_fn, action_cost_fn,
                      is_goal_fn=None):
    """general_search + expansion counter.
    Cycle prevention is handled by the queuing_fn (e.g. enqueue_at_front)."""
    if is_goal_fn is None: is_goal_fn = is_goal
    frontier = deque([make_node(problem['initial'])])
    count    = 0
    while True:
        if not frontier: return None, count
        node = frontier.popleft()
        count += 1
        if is_goal_fn(problem, node['state']): return node, count
        queuing_fn(frontier, expand(problem, node, get_actions_fn, action_cost_fn))

p = make_grid_problem((0,0),(4,4))
bfs_node, bfs_exp = search_with_count(p, enqueue_at_end,   get_actions, action_cost)
dfs_node, dfs_exp = search_with_count(p, enqueue_at_front, get_actions, action_cost)
print(f'BFS: path={reconstruct_path(bfs_node)}  nodes_expanded={bfs_exp}')
print(f'DFS: path={reconstruct_path(dfs_node)}  nodes_expanded={dfs_exp}')

In [ ]:
p = make_grid_problem((0,0),(4,4))
bfs_result = general_search(p, enqueue_at_end,   get_actions, action_cost)
dfs_result = general_search(p, enqueue_at_front, get_actions, action_cost)

fai_viz.plot_grid_path(p, reconstruct_path(bfs_result),
                       title=f"BFS — {bfs_result['depth']} steps (optimal)")
fai_viz.plot_grid_path(p, reconstruct_path(dfs_result),
                       title=f"DFS — {dfs_result['depth']} steps")

## Section 7: Apply to Water Pouring Problem

The same General-Search framework works for ANY problem — just swap the problem functions:

In [ ]:
# ── Water Pouring with General Search ────────────────────────────────────
def make_pour_problem(initial, goal, capacities):
    return {'initial': initial, 'goal': goal, 'capacities': capacities}

def pour_is_goal(problem, state):
    return problem['goal'] in state   # e.g. 7 in (0, 5, 7)

def pour_get_actions(problem, state):
    caps, n, results = problem['capacities'], len(state), []
    for i in range(n):
        if state[i] < caps[i]:
            new = list(state); new[i] = caps[i]
            results.append((f'Fill {i}', tuple(new)))
        if state[i] > 0:
            new = list(state); new[i] = 0
            results.append((f'Empty {i}', tuple(new)))
        for j in range(n):
            if i!=j and state[i]>0 and state[j]<caps[j]:
                amt=min(state[i], caps[j]-state[j])
                new=list(state); new[i]-=amt; new[j]+=amt
                results.append((f'Pour {i}->{j}', tuple(new)))
    return results

def pour_action_cost(problem, state, action):
    return 1

# Solve water jug: 3L, 5L, 9L → goal 7L  (BFS via enqueue_at_end)
pour_p = make_pour_problem((0,0,0), 7, (3,5,9))
node = general_search(pour_p, enqueue_at_end,
                      pour_get_actions, pour_action_cost, pour_is_goal)
if node:
    path = reconstruct_path(node)
    print(f'Water pouring solved in {node["depth"]} steps:')
    for s in path: print(f'  {s}')

    fai_viz.plot_jug_solution(
        path,
        pour_p['capacities'],
        title=f"BFS Water Pouring {pour_p['capacities']} → Goal: {pour_p['goal']}L",
        is_goal_fn=lambda s: pour_p['goal'] in s
    )

## Key Takeaways

1. **General-Search** is one framework — BFS and DFS differ only in QUEUING-FN
2. **BFS** (ENQUEUE-AT-END): finds shortest path (steps), explores level by level
3. **DFS** (ENQUEUE-AT-FRONT): may not find shortest path, explores deepest first
4. **Always use a is_cycle test** to avoid infinite loops on cyclic problems
5. **The same framework works for any problem** — just swap the problem functions

## 🎨 Visual: BFS vs DFS — Exploration Order

The heatmap shows the **order in which each cell was expanded** (lighter = explored earlier). BFS radiates outward in rings; DFS dives deep along one branch first.

In [ ]:
p_ord = make_grid_problem((0,0),(4,4), grid_size=5)
fai_viz.plot_bfs_dfs_exploration(
    p_ord,
    get_actions_fn=lambda s: get_actions(p_ord, s)
)